# Sybil Zero-Day — Consolidated Baselines (single source of truth)

Detectors (LOACO, benign-calibrated, source-disjoint where marked `_sdj`):
- `ae_pf`     : per-flow autoencoder (known-manifold) — clean per-flow baseline
- `ae_rate`   : source-rate autoencoder (known-manifold) — AE fails even on the good representation
- `mahal_pf`  : per-flow benign-Mahalanobis (clean)
- `mahal_rate`: source-rate benign-Mahalanobis  <-- (full + source-disjoint)
- `learned_rate`: known-attack-vs-benign direction applied to Sybil (inversion check)

Plus: operating points, source-disjoint row counts (statistical power), split-ratio robustness,
signed/artifact diagnostics, fixed raw-ID control.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from numpy.linalg import pinv
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
np.random.seed(42)
DATA="./data/"
SEEDS=(42,1,2,3,4); RS=42; FPR_GRID=(0.05,0.10,0.20)
UAV_CFG={"path":DATA+"UAVIDS-2025.csv","label_col":"label","normal":"Normal Traffic",
  "attacks":["Blackhole Attack","Flooding Attack","Sybil Attack","Wormhole Attack"],
  "leak_clean":["FlowID","SrcAddr","DstAddr","Protocol"]}
ID_CFG={"src":"SrcAddr","dst":"DstAddr"}; TARGET="Sybil Attack"
RATE=["s_fanout_rate","s_flows_per_dst","s_dst_entropy_norm"]
print("config ok")

config ok


In [ ]:
def load(cfg): return pd.read_csv(cfg["path"],low_memory=False).reset_index(drop=True)
def source_rate_features(df,idcfg):
    src=idcfg["src"]; dst=idcfg["dst"]
    s=df[src].astype(str).fillna("NA").values; dd=df[dst].astype(str).fillna("NA").values
    tmp=pd.DataFrame({"s":s,"d":dd}); g=tmp.groupby("s")
    flowcount=g["s"].transform("size").astype(float).values; fanout=g["d"].transform("nunique").astype(float).values
    ent_map={}
    for k,gg in tmp.groupby("s"):
        vc=gg["d"].value_counts().values.astype(float); p=vc/vc.sum(); ent_map[k]=float(-(p*np.log(p+1e-12)).sum())
    ent=np.array([ent_map[x] for x in s]); fc=np.clip(flowcount,1,None); fo=np.clip(fanout,1,None)
    return pd.DataFrame({"s_fanout_rate":fanout/fc,"s_flows_per_dst":flowcount/fo,
                         "s_dst_entropy_norm":ent/np.log(np.clip(fanout,2,None))}).fillna(0.0).reset_index(drop=True)
def feats_perflow(df,cfg):
    # CLEAN per-flow: drop leak_clean, label, AND any source-rate columns
    drop=set(cfg["leak_clean"])|{cfg["label_col"]}|set(RATE)
    X=df.drop(columns=[c for c in df.columns if c in drop],errors="ignore").copy()
    for c in X.columns:
        if not pd.api.types.is_numeric_dtype(X[c]): X[c]=LabelEncoder().fit_transform(X[c].astype(str))
    return X.astype(float).reset_index(drop=True)
def loaco_split(df,label_col,target,seed,val=0.3,test=0.3):
    rng=np.random.default_rng(seed)
    dft=df[df[label_col].astype(str)==target]; dfk=df[df[label_col].astype(str)!=target]
    idx=rng.permutation(len(dfk)); nv=int(len(idx)*val)
    valk=dfk.iloc[idx[:nv]]; tr=dfk.iloc[idx[nv:]]; nt=int(len(tr)*test)
    return tr.iloc[nt:], valk, pd.concat([tr.iloc[:nt],dft])
def ae_(n,seed):
    b=max(2,n//4)
    return MLPRegressor(hidden_layer_sizes=(max(8,n//2),b,max(8,n//2)),max_iter=150,early_stopping=True,n_iter_no_change=8,random_state=seed)
def ae_err(ae,X):
    r=ae.predict(X); r=r.reshape(-1,1) if r.ndim==1 else r; return np.mean((X-r)**2,1)
def fit_mahal(Xn):
    mu=Xn.mean(0); cov=np.cov(Xn.T)+1e-6*np.eye(Xn.shape[1]); return mu,pinv(cov)
def mahal(X,mu,P):
    d=X-mu; return np.einsum('ij,jk,ik->i',d,P,d)
def Z(A,Bv,Be):
    imp=SimpleImputer(strategy="mean").fit(A); sca=StandardScaler().fit(imp.transform(A))
    return sca.transform(imp.transform(A)),sca.transform(imp.transform(Bv)),sca.transform(imp.transform(Be))
print("helpers ok")

helpers ok


## Main: all detectors, one protocol

In [ ]:
def run(seeds=SEEDS,grid=FPR_GRID,val=0.3,test=0.3,verbose=True):
    cfg=UAV_CFG; idcfg=ID_CFG; lab=cfg["label_col"]; normal=cfg["normal"]; tgt=TARGET; src=idcfg["src"]
    df=load(cfg); df=pd.concat([df,source_rate_features(df,idcfg)],axis=1)
    roc={k:[] for k in ["ae_pf","ae_rate","mahal_pf","mahal_rate_full","mahal_rate_sdj","learned_rate_full","learned_rate_sdj"]}
    op=[]; sdj_n=[]; seen_frac=[]
    for seed in seeds:
        tr,valk,test_=loaco_split(df,lab,tgt,seed,val,test); test_=test_.reset_index(drop=True)
        ytr=tr[lab].astype(str).values; yv=valk[lab].astype(str).values; ye=test_[lab].astype(str).values
        is_t=(ye==tgt); is_n=(ye==normal); ka=(~is_t)&(~is_n); bmask=(ytr==normal)
        train_srcs=set(tr[src].astype(str)); seen=is_t & test_[src].astype(str).isin(train_srcs).values; sdj=~seen
        seen_frac.append(seen.sum()/max(1,is_t.sum())); sdj_n.append(int((is_t&sdj).sum()))
        # per-flow (CLEAN)
        Ap=feats_perflow(tr,cfg); Bvp=feats_perflow(valk,cfg).reindex(columns=Ap.columns,fill_value=0); Bep=feats_perflow(test_,cfg).reindex(columns=Ap.columns,fill_value=0)
        Za,Zv,Ze=Z(Ap,Bvp,Bep)
        ae=ae_(Za.shape[1],seed); ae.fit(Za,Za); roc["ae_pf"].append(roc_auc_score(is_t.astype(int),ae_err(ae,Ze)))
        mu,P=fit_mahal(Za[bmask]); roc["mahal_pf"].append(roc_auc_score(is_t.astype(int),mahal(Ze,mu,P)))
        # source-rate
        Ar=tr[RATE].reset_index(drop=True); Bvr=valk[RATE].reset_index(drop=True); Ber=test_[RATE].reset_index(drop=True)
        Za,Zv,Ze=Z(Ar,Bvr,Ber)
        ae=ae_(Za.shape[1],seed); ae.fit(Za,Za); roc["ae_rate"].append(roc_auc_score(is_t.astype(int),ae_err(ae,Ze)))
        mu,P=fit_mahal(Za[bmask]); se=mahal(Ze,mu,P); base=mahal(Zv,mu,P)[yv==normal]
        roc["mahal_rate_full"].append(roc_auc_score(is_t.astype(int),se))
        roc["mahal_rate_sdj"].append(roc_auc_score(is_t[sdj].astype(int),se[sdj]))
        yb=(ytr!=normal).astype(int); clf=LogisticRegression(max_iter=1000,class_weight="balanced").fit(Za,yb)
        sl=clf.decision_function(Ze)
        roc["learned_rate_full"].append(roc_auc_score(is_t.astype(int),sl))
        roc["learned_rate_sdj"].append(roc_auc_score(is_t[sdj].astype(int),sl[sdj]))
        opf={}
        for f in grid:
            thr=np.percentile(base,100*(1-f))
            opf[f]=dict(det=float(np.mean(se[sdj][is_t[sdj]]>thr)),
                        fpr_normal=float(np.mean(se[is_n]>thr)),
                        fpr_known=float(np.mean(se[ka]>thr)))
        op.append(opf)
    if verbose:
        print(f"source-disjoint Sybil rows/seed: mean={np.mean(sdj_n):.0f} (min {min(sdj_n)})  "
              f"| seen-source frac={np.mean(seen_frac)*100:.1f}%\n")
        print("=== ROC (LOACO, mean\u00b1std) ===")
        for k in roc: v=np.array(roc[k]); print(f"  {k:18s} {v.mean():.3f}\u00b1{v.std():.3f}")
        print("\n=== mahal_rate operating points (benign-calibrated, source-disjoint det) ===")
        for f in grid:
            D=pd.DataFrame([o[f] for o in op]); m=D.mean()
            print(f"  fpr={f:.2f} -> det={m['det']:.3f}  fpr_normal={m['fpr_normal']:.3f}  fpr_known={m['fpr_known']:.3f}")
    return {k:(np.mean(v),np.std(v)) for k,v in roc.items()}
RES=run()

source-disjoint Sybil rows/seed: mean=21128 (min 20553)  | seen-source frac=12.2%

=== ROC (LOACO, mean±std) ===
  ae_pf              0.358±0.117
  ae_rate            0.857±0.121
  mahal_pf           0.804±0.010
  mahal_rate_full    0.888±0.002
  mahal_rate_sdj     0.907±0.002
  learned_rate_full  0.087±0.000
  learned_rate_sdj   0.027±0.001

=== mahal_rate operating points (benign-calibrated, source-disjoint det) ===
  fpr=0.05 -> det=0.786  fpr_normal=0.049  fpr_known=0.277
  fpr=0.10 -> det=0.925  fpr_normal=0.099  fpr_known=0.350
  fpr=0.20 -> det=1.000  fpr_normal=0.196  fpr_known=0.486


## Robustness: split-ratio (source-rate has NO window hyperparameter)

The proposed source-rate representation has no window size to tune. We instead vary the
(val,test) split fractions to show `mahal_rate_sdj` is stable.

In [ ]:
print("split-ratio robustness (mahal_rate_sdj ROC):")
for v,t in [(0.3,0.3),(0.2,0.2),(0.4,0.2),(0.2,0.4)]:
    r=run(val=v,test=t,verbose=False)["mahal_rate_sdj"]
    print(f"  val={v},test={t} -> {r[0]:.3f}\u00b1{r[1]:.3f}")

split-ratio robustness (mahal_rate_sdj ROC):
  val=0.3,test=0.3 -> 0.907±0.002
  val=0.2,test=0.2 -> 0.912±0.006
  val=0.4,test=0.2 -> 0.907±0.003
  val=0.2,test=0.4 -> 0.907±0.002


## Diagnostics: signed/artifact AUC + fixed raw-ID control

"Source-disjoint" testinde, test setindeki Sybil akışlarının bir kısmının kaynağı (SrcAddr) eğitim setinde de görülmüş oluyor. Bunları dışlıyoruz ki dedektör kimliği ezberleyip puan şişirmesin — yalnız görülmemiş kaynaklardaki Sybil'de ölçüyoruz. "~%12" = bu dışlanan (görülmüş-kaynaklı) Sybil oranı.

In [ ]:
cfg=UAV_CFG; idcfg=ID_CFG; lab=cfg["label_col"]; normal=cfg["normal"]
df=load(cfg); F=source_rate_features(df,idcfg); y=(df[lab].astype(str).values==TARGET).astype(int)
print("Signed per-feature AUC (raw) on source-rate features:")
best=0
for c in F.columns:
    a=roc_auc_score(y,F[c].values); best=max(best,max(a,1-a))
    print(f"  {c:18s} raw_AUC={a:.3f} ({'Sybil HIGH' if a>0.5 else 'Sybil LOW'})  sep={max(a,1-a):.3f}")
print(f"[artifact] best source-rate single-feature AUC={best:.3f} -> {'ARTIFACT' if best>0.99 else 'not trivial (<0.99)'}")
# fixed raw-ID control (target vs normal, seen-only)
dff=pd.concat([df,F],axis=1); tr,valk,test_=loaco_split(dff,lab,TARGET,RS); test_=test_.reset_index(drop=True)
rate=tr.assign(_a=(tr[lab].astype(str)!=normal).astype(int)).groupby("SrcAddr")["_a"].mean()
sub=test_[test_[lab].astype(str).isin([normal,TARGET])].copy(); sub["_seen"]=sub["SrcAddr"].astype(str).isin(rate.index)
syb=sub[sub[lab].astype(str)==TARGET]
print(f"\n[raw-ID] Sybil rows with SEEN source: {syb['_seen'].mean()*100:.1f}%  (unseen cannot be scored by identity)")

Signed per-feature AUC (raw) on source-rate features:
  s_fanout_rate      raw_AUC=0.219 (Sybil LOW)  sep=0.781
  s_flows_per_dst    raw_AUC=0.781 (Sybil HIGH)  sep=0.781
  s_dst_entropy_norm raw_AUC=0.811 (Sybil HIGH)  sep=0.811
[artifact] best source-rate single-feature AUC=0.811 -> not trivial (<0.99)

[raw-ID] Sybil rows with SEEN source: 10.7%  (unseen cannot be scored by identity)


## Locked numbers for the manuscript
After running, the values above are the SINGLE source for all baseline/headline numbers.
`ae_pf`/`mahal_pf` here are the CLEAN per-flow baselines (v3's were contaminated).

# Saldırı-başına nokta-temelli OOD

In [ ]:
# === Motivation: per-attack zero-day with point-wise novelty scores (single source of truth) ===
# AE-known (known-manifold AE) | Energy | class-conditional Mahalanobis (Lee-style)
from numpy.linalg import pinv as _pinv
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LogisticRegression

# AE yardimcilari
def _ae(n, seed):
    b=max(2, n//4)
    return MLPRegressor(hidden_layer_sizes=(max(8,n//2), b, max(8,n//2)),
                        max_iter=150, early_stopping=True, n_iter_no_change=8, random_state=seed)
def _ae_err(ae, X):
    r=ae.predict(X); r=r.reshape(-1,1) if r.ndim==1 else r; return np.mean((X-r)**2, 1)

def _energy_score(logits, T=1.0):
    m=logits.max(1, keepdims=True)
    lse=(m.squeeze(1)+np.log(np.exp((logits-m)/T).sum(1))*T)
    return -(T*lse)          # HIGH = OOD
def _fit_cc_mahal(X, y):
    classes=np.unique(y); mus={}; d=X.shape[1]; S=np.zeros((d,d)); n=0
    for c in classes:
        Xc=X[y==c]; mu=Xc.mean(0); mus[c]=mu
        S+=((Xc-mu).T@(Xc-mu)); n+=len(Xc)
    P=_pinv(S/max(1,n)+1e-6*np.eye(d)); return mus,P,classes
def _cc_mahal_score(X, mus, P, classes):
    dists=np.stack([np.einsum('ij,jk,ik->i', X-mus[c], P, X-mus[c]) for c in classes], 1)
    return dists.min(1)       # HIGH = far from all known = OOD

def _Z(A, Be):               # impute+scale on A (train), apply to Be (test)
    imp=SimpleImputer(strategy="mean").fit(A); sc=StandardScaler().fit(imp.transform(A))
    return sc.transform(imp.transform(A)), sc.transform(imp.transform(Be))

def per_attack_pointwise(seeds=SEEDS):
    cfg=UAV_CFG; lab=cfg["label_col"]; normal=cfg["normal"]
    df=load(cfg)
    attacks=list(cfg["attacks"])
    rows={a:{"AE-known":[], "Energy":[], "Mahalanobis(cc)":[]} for a in attacks}
    for tgt in attacks:
        for seed in seeds:
            tr,valk,test_=loaco_split(df,lab,tgt,seed); test_=test_.reset_index(drop=True)
            ytr=tr[lab].astype(str).values; ye=test_[lab].astype(str).values
            is_t=(ye==tgt).astype(int)
            A=feats_perflow(tr,cfg); Be=feats_perflow(test_,cfg).reindex(columns=A.columns,fill_value=0)
            Za,Ze=_Z(A,Be)
            # AE-known
            ae=_ae(Za.shape[1],seed); ae.fit(Za,Za)
            rows[tgt]["AE-known"].append(roc_auc_score(is_t,_ae_err(ae,Ze)))
            # Energy + cc-Mahalanobis (supervised backbone on known classes)
            clf=LogisticRegression(max_iter=1000, multi_class="multinomial").fit(Za,ytr)
            logits=clf.decision_function(Ze); logits=logits if logits.ndim==2 else np.c_[-logits,logits]
            rows[tgt]["Energy"].append(roc_auc_score(is_t,_energy_score(logits)))
            mus,P,classes=_fit_cc_mahal(Za,ytr)
            rows[tgt]["Mahalanobis(cc)"].append(roc_auc_score(is_t,_cc_mahal_score(Ze,mus,P,classes)))
    print("=== Per-attack zero-day ROC (point-wise novelty, LOACO, mean\u00b1std) ===")
    print(f"{'attack':18s} {'AE-known':>14} {'Energy':>14} {'Mahalanobis(cc)':>16} {'best':>7}")
    summary={}
    for a in attacks:
        vals={m:np.array(rows[a][m]) for m in rows[a]}
        best=max(v.mean() for v in vals.values()); summary[a]=best
        print(f"{a:18s} "+" ".join(f"{vals[m].mean():.3f}\u00b1{vals[m].std():.3f}"
              for m in ['AE-known','Energy','Mahalanobis(cc)'])+f"   {best:.3f}")
    print("\nReading: every attack except Sybil has a point-wise score with high ROC; Sybil's best is low.")
    print("Cross-check: AE-known(Sybil) should match ae_pf (~0.358).")
    return rows, summary

ROWS_PA, BEST_PA = per_attack_pointwise()

=== Per-attack zero-day ROC (point-wise novelty, LOACO, mean±std) ===
attack                   AE-known         Energy  Mahalanobis(cc)    best
Blackhole Attack   0.819±0.033 0.309±0.014 0.767±0.003   0.819
Flooding Attack    0.816±0.066 0.198±0.034 0.801±0.004   0.816
Sybil Attack       0.358±0.117 0.483±0.005 0.477±0.005   0.483
Wormhole Attack    0.655±0.067 0.819±0.001 0.637±0.002   0.819

Reading: every attack except Sybil has a point-wise score with high ROC; Sybil's best is low.
Cross-check: AE-known(Sybil) should match ae_pf (~0.358).


# Per-flow single artifact AUC

In [ ]:
# === Per-flow single-feature artifact AUC (Sybil vs Normal) — diagnostic for §4.1 ===
def perflow_artifact_auc():
    cfg=UAV_CFG; lab=cfg["label_col"]; normal=cfg["normal"]; tgt=TARGET
    df=load(cfg)
    Xall=feats_perflow(df,cfg)                 # clean per-flow (excludes RATE+leak+label)
    mask=df[lab].astype(str).isin([normal,tgt]).values
    X=Xall[mask].values; y=(df[lab].astype(str)[mask].values==tgt).astype(int)
    aucs=sorted(((max(a:=roc_auc_score(y,X[:,j]),1-a), c) for j,c in enumerate(Xall.columns)), reverse=True)
    print("Per-flow single-feature AUC (Sybil vs Normal), top 5:")
    for a,c in aucs[:5]: print(f"  {c:22s} {a:.3f}")
    print(f"[per-flow artifact] BEST = {aucs[0][0]:.3f}  | >0.95: {sum(a>0.95 for a,_ in aucs)}/{len(aucs)}")
    return aucs[0]
perflow_artifact_auc()

Per-flow single-feature AUC (Sybil vs Normal), top 5:
  TxByteRate/s           0.994
  TxPacketRate/s         0.990
  RxByteRate/s           0.987
  Throughput/Kbps        0.987
  RxPacketRate/s         0.980
[per-flow artifact] BEST = 0.994  | >0.95: 7/18


(np.float64(0.9941468165609659), 'TxByteRate/s')

# RF Gate

In [ ]:
# === RF-confidence gate (cascade diagnosis): single source of truth ===
from sklearn.ensemble import RandomForestClassifier

def rf_gate(seeds=SEEDS, fpr_target=0.10):
    cfg=UAV_CFG; lab=cfg["label_col"]; normal=cfg["normal"]; tgt=TARGET
    df=load(cfg)
    pass_n=[]; pass_k=[]; pass_t=[]
    for seed in seeds:
        tr,valk,test_=loaco_split(df,lab,tgt,seed); test_=test_.reset_index(drop=True)
        ytr=tr[lab].astype(str).values; ye=test_[lab].astype(str).values; yv=valk[lab].astype(str).values
        A=feats_perflow(tr,cfg)
        Be=feats_perflow(test_,cfg).reindex(columns=A.columns,fill_value=0)
        Bv=feats_perflow(valk,cfg).reindex(columns=A.columns,fill_value=0)
        Za,Ze=_Z(A,Be) if '_Z' in globals() else (None,None)
        if Za is None:  # fallback if _Z not defined
            imp=SimpleImputer(strategy="mean").fit(A); sc=StandardScaler().fit(imp.transform(A))
            Za=sc.transform(imp.transform(A)); Ze=sc.transform(imp.transform(Be)); Zv=sc.transform(imp.transform(Bv))
        else:
            imp=SimpleImputer(strategy="mean").fit(A); sc=StandardScaler().fit(imp.transform(A))
            Zv=sc.transform(imp.transform(Bv))
        rf=RandomForestClassifier(n_estimators=200,random_state=seed,n_jobs=1).fit(Za,ytr)
        conf=rf.predict_proba(Ze).max(1); confv=rf.predict_proba(Zv).max(1)
        tau=np.percentile(confv[yv==normal], 100*fpr_target)   # benign-calibrated low-confidence cutoff
        fwd=conf<tau
        is_t=(ye==tgt); is_n=(ye==normal); ka=(~is_t)&(~is_n)
        pass_n.append(fwd[is_n].mean()); pass_k.append(fwd[ka].mean()); pass_t.append(fwd[is_t].mean())
    print(f"RF-confidence gate (forward = below benign-calibrated {fpr_target:.0%} confidence cutoff):")
    print(f"  pass-rate normal       = {np.mean(pass_n):.3f}")
    print(f"  pass-rate known-attack = {np.mean(pass_k):.3f}")
    print(f"  pass-rate Sybil        = {np.mean(pass_t):.3f}  <-- forwarded to novelty detector")
    print(f"  => {(1-np.mean(pass_t))*100:.1f}% of unseen Sybil confidently absorbed as a known class")
    return np.mean(pass_t)

rf_gate()

RF-confidence gate (forward = below benign-calibrated 10% confidence cutoff):
  pass-rate normal       = 0.094
  pass-rate known-attack = 0.293
  pass-rate Sybil        = 0.034  <-- forwarded to novelty detector
  => 96.6% of unseen Sybil confidently absorbed as a known class


np.float64(0.03357561157951572)

# Write to CSV

In [ ]:
# === Fig.2 ROC curves -> fig2_roc.csv (FinalBaselines: ae_pf, mahal_pf, mahal_rate) ===
import os, json, numpy as np, pandas as pd
from sklearn.metrics import roc_curve
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor

def _scale2(A,Be):
    imp=SimpleImputer(strategy="mean").fit(A); sc=StandardScaler().fit(imp.transform(A))
    return sc.transform(imp.transform(A)), sc.transform(imp.transform(Be))
def _ae2(n,seed):
    b=max(2,n//4)
    return MLPRegressor(hidden_layer_sizes=(max(8,n//2),b,max(8,n//2)),max_iter=150,
                        early_stopping=True,n_iter_no_change=8,random_state=seed)

def fig2_save_main(seeds=SEEDS, csv="fig2_roc.csv"):
    grid=np.linspace(0,1,101)
    cfg=UAV_CFG; lab=cfg["label_col"]; normal=cfg["normal"]; tgt=TARGET; src=ID_CFG["src"]
    df=load(cfg); df=pd.concat([df,source_rate_features(df,ID_CFG)],axis=1)
    store={d:[] for d in ["ae_pf","mahal_pf","mahal_rate"]}
    for seed in seeds:
        tr,valk,test_=loaco_split(df,lab,tgt,seed); test_=test_.reset_index(drop=True)
        ytr=tr[lab].astype(str).values; ye=test_[lab].astype(str).values
        is_t=(ye==tgt); bmask=(ytr==normal)
        seen=is_t & test_[src].astype(str).isin(set(tr[src].astype(str))).values; sdj=~seen
        yb=is_t[sdj].astype(int)
        # per-flow AE
        Ap=feats_perflow(tr,cfg); Bep=feats_perflow(test_,cfg).reindex(columns=Ap.columns,fill_value=0)
        Za,Ze=_scale2(Ap,Bep)
        ae=_ae2(Za.shape[1],seed); ae.fit(Za,Za); r=ae.predict(Ze); r=r.reshape(-1,1) if r.ndim==1 else r
        e=np.mean((Ze-r)**2,1)
        f,t,_=roc_curve(yb,e[sdj]); store["ae_pf"].append(np.interp(grid,f,t))
        # per-flow benign-Mahalanobis (artifact)
        mu,P=fit_mahal(Za[bmask]); s=mahal(Ze,mu,P)
        f,t,_=roc_curve(yb,s[sdj]); store["mahal_pf"].append(np.interp(grid,f,t))
        # source-rate benign-Mahalanobis (ours)
        Ar=tr[RATE].reset_index(drop=True); Ber=test_[RATE].reset_index(drop=True)
        Za2,Ze2=_scale2(Ar,Ber); mu,P=fit_mahal(Za2[bmask]); s=mahal(Ze2,mu,P)
        f,t,_=roc_curve(yb,s[sdj]); store["mahal_rate"].append(np.interp(grid,f,t))
    rows=[]
    for d,arr in store.items():
        A=np.vstack(arr)
        rows+=[{"detector":d,"fpr":float(x),"tpr_mean":float(m),"tpr_std":float(sd)}
               for x,m,sd in zip(grid,A.mean(0),A.std(0))]
    new=pd.DataFrame(rows)
    if os.path.exists(csv):
        old=pd.read_csv(csv); old=old[~old["detector"].isin(store.keys())]; new=pd.concat([old,new],ignore_index=True)
    new.to_csv(csv,index=False); print(f"wrote {csv} | detectors now:", sorted(new.detector.unique()))
fig2_save_main()